# 01 · Exploration du dataset

Dataset : **Fashion Product Images (Small)** (Kaggle). Objectif de ce notebook : comprendre les données **avant** de choisir
les classes à prédire (type de vêtement, couleur, style) et repérer les problèmes (classes rares, déséquilibre, qualité des images).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, "..")                    # pour importer les modules de ml/
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("../dataset")                   # dossier contenant styles.csv et images/

from preprocess import build_dataframe
raw = pd.read_csv(next(DATA.rglob("styles.csv")), on_bad_lines="skip")
print("Lignes dans styles.csv :", len(raw))
raw.head()

## Classes retenues pour l'application
On ne garde que les types de vêtements utiles au dressing, les couleurs assez fréquentes et les 3 styles (Casual, Formal, Sports).

In [ ]:
df, images = build_dataframe(DATA, min_color_count=200)
print(len(df), "images retenues sur", len(raw))
print("types :", df.articleType.nunique(), "| couleurs :", df.baseColour.nunique(), "| styles :", df.usage.nunique())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, title) in zip(axes, [("articleType", "Type de vêtement"), ("baseColour", "Couleur"), ("usage", "Style")]):
    df[col].value_counts().sort_values().plot.barh(ax=ax, color="#4f7cff")
    ax.set_title(title); ax.set_xlabel("nombre d'images")
plt.tight_layout(); plt.show()

### À observer
- Les classes sont **déséquilibrées** (beaucoup de T-shirts et de chaussures casual, peu de ballerines ou de jupes) : on utilisera `class_weight="balanced"` et on regardera le **F1 macro** (qui donne le même poids à chaque classe), pas seulement l'accuracy.
- Certaines classes se ressemblent (Pantalon / Jogging, Chemise / Top, Pull / Sweat) : on s'attend à des confusions entre elles.

## Quelques images par type

In [ ]:
from PIL import Image
types = sorted(df.articleType.unique())
fig, axes = plt.subplots(len(types), 5, figsize=(8, 1.7 * len(types)))
for row, t in enumerate(types):
    sample = df[df.articleType == t].sample(5, random_state=0)
    for col, (_, r) in enumerate(sample.iterrows()):
        axes[row, col].imshow(Image.open(images / f"{r.id}.jpg")); axes[row, col].axis("off")
    axes[row, 0].set_title(t, loc="left", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
sizes = pd.Series([Image.open(images / f"{i}.jpg").size for i in df.id.sample(300, random_state=0)]).value_counts()
print("Tailles des images (largeur, hauteur) :"); print(sizes.head())

**Remarque importante pour la suite :** les images sont petites (60 × 80 pixels environ), sur fond clair, un seul vêtement par image.
Les photos de l'étudiant devront ressembler à cela (un vêtement, fond uni) ; c'est pourquoi l'extraction de caractéristiques
recadre le vêtement et réduit la photo à la même taille avant de calculer quoi que ce soit.